In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualizacao grafica inline no Jupyter notebook
%matplotlib inline

# Carregar uma gravacao de EEG

**Dificuldade 1** | **Tempo de execucao: 2m** | **Computacao: CPU**

Depois que uma busca retorna um conjunto de dados candidato (veja plot_00), a proxima pergunta e pratica: o que *uma* gravacao realmente contem? Este tutorial carrega um unico arquivo BIDS do [OpenNeuro](https://openneuro.org) ``ds004504`` (Miltiadous et al. 2023; Alzheimer / demencia frontotemporal / controles saudaveis) por meio do catalogo compartilhado com o [NEMAR](https://nemar.org) :cite:`delorme2022nemar`, descompacta o objeto :class:`mne.io.Raw` e inspeciona canais, montagem e espectro. O conjunto de dados e pequeno (88 sujeitos, ~10 min por sujeito, 19 canais a 500 Hz em um layout padrao 10-20), de modo que o primeiro download e inferior a 25 MB e todas as etapas subsequentes leem diretamente do cache local.

.. sphinx_gallery_thumbnail_path = '_static/thumbs/plot_01_first_recording.png'
Palavras-chave: carregamento, BIDS


## Objetivos de aprendizagem
- Construir um :class:`~eegdash.api.EEGDashDataset` e ler sua representacao HTML.
- Selecionar um registro indexando em ``dataset.datasets``.
- Ler taxa de amostragem, contagem de canais e duracao a partir do :class:`mne.io.Raw` subjacente.
- Visualizar o sinal (``raw.plot``), a montagem de eletrodos (``raw.plot_sensors``) e o espectro (``raw.compute_psd().plot``).



## Requisitos
- Cerca de 2 min em CPU na primeira execucao; menos de 10 s uma vez em cache.
- Rede: ~80 MB na primeira chamada (baixados uma unica vez para o cache).
- Diretorio de cache: variavel de ambiente ``EEGDASH_CACHE_DIR``, com padrao em ``~/.eegdash_cache``.
- Pre-requisito: ``plot_00_first_search``.



Configuracao inicial. Sem aleatoriedade aqui, portanto sem semente (*seed*).



In [ ]:
# Importa utilitarios de sistema e manipulacao de diretorios
import os
from pathlib import Path

# Importa bibliotecas para plotagem grafica, MNE para dados eletrofisiologicos e arrays numericos
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

# Importa o modulo eegdash e o construtor de datasets
import eegdash
from eegdash import EEGDashDataset
# Importa estilos visuais padronizados do EEGDash
from eegdash.viz import style_figure, use_eegdash_style

# Forca o backend matplotlib para o navegador do MNE para permitir captura estatica das figuras
mne.viz.set_browser_backend("matplotlib")

# Aplica o tema visual do EEGDash
use_eegdash_style()
# Define o caminho do diretorio de cache persistente
cache_dir = os.environ.get("EEGDASH_CACHE_DIR", str(Path.home() / ".eegdash_cache"))
print(f"eegdash {eegdash.__version__}; cache_dir={cache_dir}")

## Conceitos por tras de ``EEGDashDataset``
Algumas ideias que o restante do tutorial pressupoe:

- **Indice de metadados vs. arquivos brutos.** O EEGDash separa *registros* (pequenos documentos JSON no MongoDB) de *arquivos* (cargas uteis BIDS no OpenNeuro / S3). Uma consulta toca apenas o indice; os bytes de sinal permanecem remotos ate serem solicitados.
- **Preguicoso (*lazy*) por padrao.** ``EEGDashDataset(...)`` retorna imediatamente; nenhum sinal trafega pela rede nesse momento. Cada entrada por registro expoe ``raw`` como uma *propriedade*, portanto o download e a abertura ocorrem na primeira vez em que voce o le (e apenas para aquele registro especifico). O construtor aceita ``download=False`` para impor modo estritamente offline contra um cache ja existente; caso contrario, o EEGDash busca arquivos faltantes no ``cache_dir`` sob demanda. ``download_all()`` permite carregar tudo previamente de forma explicita quando um pipeline nao pode lidar com downloads em tempo de execucao.
- **Heranca do Braindecode.** O ``EEGDashDataset`` e uma subclasse de :class:`braindecode.datasets.BaseConcatDataset`; cada entrada em :attr:`~eegdash.api.EEGDashDataset.datasets` e uma subclasse de :class:`braindecode.datasets.BaseDataset`. Qualquer codigo compativel com esses tipos (``create_fixed_length_windows(ds, ...)``, ``DataLoader(ds, ...)``, pipelines ``Preprocessor(...)``) aceita um EEGDashDataset sem modificacoes.
- **Envio e recuperacao do Hub.** :meth:`~eegdash.api.EEGDashDataset.push_to_hub` / ``pull_from_hub`` realizam o ciclo completo de um conjunto de dados derivado (sinais pre-processados, janelas ou tabelas de caracteristicas) com um repositorio no HuggingFace Hub. O indice de metadados continua sendo a fonte de verdade para as gravacoes originais; o Hub e destinado a artefatos versionados e prontos para aprendizado de maquina construidos *a partir* deles.
- **Entidades BIDS sao a linguagem de consulta.** Parametros como ``dataset``, ``subject``, ``task``, ``session``, ``run`` sao passados diretamente para o indice sem alteracoes :cite:`pernet2019eegbids`.

## Etapa 1: Construir o dataset (carregamento preguicoso / lazy)
A construcao de um :class:`~eegdash.api.EEGDashDataset` apenas consulta o catalogo de metadados. Nenhum byte de sinal de EEG e transferido ainda.

**Preveja.** Quantos registros a consulta para ``ds004504`` sujeito ``001`` retornara: 1 arquivo de olhos fechados, varias tarefas ou todo o conjunto de dados?

**Execute.** Construa o objeto; a representacao HTML abaixo mostra o que foi encontrado.



In [ ]:
# Define identificadores do conjunto de dados e do participante desejado
DATASET = "ds004504"  # Miltiadous AD/FTD/HC, 19 canais padronizados 10-20, 500 Hz
SUBJECT = "001"
# Cria o dataset preguicoso consultando o catalogo do EEGDash
dataset = EEGDashDataset(cache_dir=cache_dir, dataset=DATASET, subject=SUBJECT)
# Renderiza a representacao tabular do BaseConcatDataset
dataset

## Etapa 2: Resumir sem acessar a rede
Cada registro compativel expoe seus metadados BIDS em :attr:`~eegdash.api.EEGDashDataset.description`, um :class:`pandas.DataFrame` materializado sem qualquer chamada de rede. Util para verificar a sanidade de uma consulta antes de assumir custos de download.



In [ ]:
# Extrai o DataFrame de metadados das gravacoes do dataset
description = dataset.description
# Obtem o primeiro registro de metadados brutos se disponivel
first_record = dataset.records[0] if getattr(dataset, "records", None) else {}


# Funcao auxiliar para contar valores unicos em colunas presentes
def _nunique(col: str) -> int:
    return description[col].nunique() if col in description.columns else 0


# Constroi e exibe resumo quantitativo de registros, sujeitos, tarefas e canais
pd.Series(
    {
        "n_records": len(dataset),
        "n_subjects": _nunique("subject"),
        "n_tasks": _nunique("task"),
        "n_sessions": _nunique("session"),
        "channels": int(first_record.get("nchans", 0)),
        "sampling_frequency_hz": float(first_record.get("sampling_frequency", 0.0)),
    },
    name="value",
).to_frame()

## Etapa 3: Selecionar um registro
:class:`~eegdash.api.EEGDashDataset` encapsula uma lista de entradas por gravacao em :attr:`~eegdash.api.EEGDashDataset.datasets`. Cada entrada e um :class:`EEGDashBaseDataset` (uma subclasse de :class:`BaseDataset` do Braindecode) contendo um caminho BIDS, uma descricao e a propriedade preguicosa ``raw``. Indexar nessa lista e o padrao idiomatico do Python para selecionar um item; ``record.raw`` e o que dispara o download e abre o arquivo com o MNE-Python :cite:`gramfort2013mne`.



In [ ]:
# Seleciona a primeira gravacao da lista de datasets
record = dataset.datasets[0]
# Acessa o atributo raw, disparando o download do arquivo de sinal se necessario e abrindo-o com MNE
raw = record.raw

## Etapa 4: Metadados BIDS como um DataFrame
Toda gravacao carrega o mesmo envelope BIDS :cite:`pernet2019eegbids`. ``record.description`` ja e uma :class:`pandas.Series`; manter os campos de interesse e adicionar a duracao fisica fornece uma visualizacao legivel.



In [ ]:
# Define campos BIDS prioritarios para inspecao
keep = [
    "dataset",
    "subject",
    "task",
    "session",
    "run",
    "sampling_frequency",
    "nchans",
    "ntimes",
    "datatype",
]
# Filtra e reindexa os metadados da gravacao em uma tabela
meta_view = record.description.reindex(keep).to_frame("value")
# Calcula e insere a duracao temporal real em segundos a partir do vetor de tempos do MNE
meta_view.loc["duration (s)"] = round(raw.times[-1] - raw.times[0], 1)
# Informa a contagem de anotacoes registradas na gravacao
meta_view.loc["annotations"] = len(raw.annotations)
# Renderiza os metadados consolidados
meta_view

## Etapa 5: Inspecionar o objeto Raw do MNE
O MNE-Python fornece sua propria representacao HTML para o objeto :class:`~mne.io.Raw`: uma tabela informativa com taxa de amostragem, projecoes, canais ruins (*bad channels*) e tipos de canais.

**Investigue.** Observe a taxa, a divisao dos tipos de canais e confira a linha "bad channels": vazia aqui (nenhum canal marcado como ruim inicialmente).



In [ ]:
# Exibe a representacao interativa padrao do objeto Raw do MNE
raw

## Etapa 6: Plotar o sinal
**Execute.** O metodo :meth:`mne.io.Raw.plot` renderiza uma janela de 8 s sobre uma selecao representativa de canais da linha media + temporais, oferecendo a visao mais legivel nesta escala. Configurar explicitamente ``scalings`` e uma ``duration`` maior torna a morfologia do sinal visivel; os padroes automaticos tendem a comprimir muitos canais na mesma faixa e achatar o tracado.



In [ ]:
# Seleciona ate 8 canais da linha media e temporais presentes na gravacao para inspecao visual
midline_picks = [
    ch
    for ch in ("Fz", "Cz", "Pz", "Oz", "FCz", "CPz", "POz", "T7", "T8")
    if ch in raw.ch_names
][:8]
# Renderiza grafico de tracado continuo do sinal para o intervalo de 10 a 18 segundos
fig_raw = raw.plot(
    start=10.0,
    duration=8.0,
    picks=midline_picks if midline_picks else None,
    n_channels=8,
    scalings={"eeg": 50e-6},
    show=False,
    show_scrollbars=False,
    show_scalebars=False,
    title="",
)

## Etapa 7: Topologia dos sensores
A localizacao dos eletrodos no escalpo e fundamental para todas as analises subsequentes. O metodo :meth:`mne.io.Raw.plot_sensors` desenha a montagem em um esquema 2D da cabeca. Com muitos canais, sobrepor os nomes de todos eles sobrecarrega a visualizacao; aqui omitimos os nomes para enfatizar a distribuicao espacial.



O dataset ds004504 usa o layout padrao 10-20 (Fp1, Fp2, F3, F4, ... Cz, Pz, 19 eletrodos); a montagem ``standard_1020`` cobre todos eles. A chamada ``plot_sensors`` so e renderizada se as posicoes forem de fato vinculadas, mantendo a celula robusta caso ocorram nomenclaturas arbitrarias de canais.



In [ ]:
# Associa a montagem com as coordenadas 3D padrao do sistema internacional 10-20
raw.set_montage(
    "standard_1020",
    match_case=False,
    match_alias=True,
    on_missing="ignore",
)
# Verifica se as posicoes dos sensores foram definidas no objeto de informacao
has_positions = any(not np.allclose(ch["loc"][:3], 0.0) for ch in raw.info["chs"])
fig_sens, ax_sens = plt.subplots(figsize=(5.0, 5.0))
# Se houver coordenadas espaciais validas, plota o mapa topografico dos sensores
if has_positions:
    raw.plot_sensors(
        kind="topomap",
        show_names=False,
        axes=ax_sens,
        show=False,
    )
    ax_sens.set_title("")
else:
    # Mensagem de contingencia caso a montagem nao tenha atribuido coordenadas
    ax_sens.text(
        0.5,
        0.5,
        "Sensor positions unavailable in this build's montage",
        ha="center",
        va="center",
    )
    ax_sens.set_axis_off()
# Ajusta o layout do grafico
fig_sens.tight_layout()

## Etapa 8: Densidade espectral de potencia (PSD)
A PSD e a melhor visualizacao inicial da qualidade do sinal: ruido de rede eletrica manifesta-se em picos em 50 ou 60 Hz, derivas lentas abaixo de 1 Hz e o decaimento classico 1/f em banda larga. Calculamos a PSD de Welch apenas sobre os canais de EEG e plotamos em decibeis (dB).



In [ ]:
# Copia o sinal bruto, seleciona canais de EEG e calcula o espectro Welch ate 80 Hz
psd = raw.copy().pick("eeg").compute_psd(fmax=80.0, verbose=False)
# Plota o espectro medio entre os canais de EEG
fig_psd = psd.plot(picks="eeg", average=True, show=False)
# Aplica estilizacao de cabecalhos e proveniencia grafica ao grafico de PSD
style_figure(
    fig_psd,
    title="Power spectral density (Welch)",
    subtitle=f"{DATASET} sub-{SUBJECT} | {len(raw.copy().pick('eeg').ch_names)} EEG channels | fmax=80 Hz",
    source=f"EEGDash plot_01 | OpenNeuro {DATASET} :cite:`miltiadous2023`",
)
# Exibe a figura gerada
plt.show()

## Um erro comum e como se recuperar
**Execute.** Indexar alem do tamanho de ``dataset.datasets`` levanta a excecao :class:`IndexError`, o contrato padrao do Python. Nos a disparamos propositalmente para que o modo de falha fique evidente :cite:`nederbragt2020teaching`.



In [ ]:
# Demonstracao de captura de erro ao tentar indexar gravacao inexistente
try:
    _ = dataset.datasets[999]
except IndexError as exc:
    print(f"Caught IndexError: {exc}")
    # Recuperacao: limita o indice ao ultimo elemento valido da lista
    safe_idx = min(999, len(dataset.datasets) - 1)
    print(f"Recovery: dataset.datasets[{safe_idx}] instead.")

## Modifique
**Sua vez.** Altere ``SUBJECT`` para um sujeito diferente e execute a Etapa 3 novamente. O protocolo de aquisicao e compartilhado, portanto a taxa de amostragem e a contagem de canais coincidira; pequenas diferencas na duracao da gravacao por sujeito sao normais.



In [ ]:
# Carrega um segundo sujeito para comparar caracteristicas tecnicas
SECOND_SUBJECT = "002"
raw_b = (
    EEGDashDataset(cache_dir=cache_dir, dataset=DATASET, subject=SECOND_SUBJECT)
    .datasets[0]
    .raw
)
# Monta tabela comparando parametros fundamentais entre os dois sujeitos
pd.DataFrame(
    {
        f"sub-{SUBJECT}": [
            raw.info["sfreq"],
            raw.info["nchan"],
            round(raw.times[-1], 1),
        ],
        f"sub-{SECOND_SUBJECT}": [
            raw_b.info["sfreq"],
            raw_b.info["nchan"],
            round(raw_b.times[-1], 1),
        ],
    },
    index=["sfreq (Hz)", "nchan", "duration (s)"],
)

## Crie
**Mini-projeto.** Escolha um conjunto de dados diferente do OpenNeuro (comece com ``ds002893`` para um estudo auditivo de 32 canais) e repita as Etapas 1, 3 e 5. Compare a montagem e a forma da PSD.



## Conclusao
Carregamos uma gravacao de ponta a ponta sem precisar escrever um script de download: :class:`~eegdash.api.EEGDashDataset` resolveu a consulta BIDS contra o catalogo NEMAR / OpenNeuro, ``dataset.datasets[0].raw`` materializou o arquivo sob demanda e o MNE-Python conduziu o restante. O sinal esta bruto; ruido de linha e derivas ainda estao visiveis no tracado. O pre-processamento completo e abordado em ``plot_10_preprocess_and_window``. A seguir: ``plot_02_dataset_to_dataloader`` encapsula essas gravacoes em um ``DataLoader`` do PyTorch.



## Referencias
Consulte :doc:`/references` para a bibliografia centralizada dos artigos citados acima. Adicione ou altere uma entrada uma unica vez em :file:`docs/source/refs.bib`; todos os tutoriais herdam a atualizacao.

